In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

print("library import is completed")

# 1. データの読み込み
path = "/kaggle/input/competitions/playground-series-s6e3/"
train_df = pd.read_csv(path + "train.csv")
test_df = pd.read_csv(path + "test.csv")

print("Data import is completed")

# ==========================================
# 徹底的な特徴量エンジニアリング
# ==========================================
def advanced_feature_engineering(df):
    df = df.copy()
    
    # --- 1. 家族構成・デモグラフィック ---
    # パートナーや扶養家族がいるか（ロックインされやすいか）
    df['Is_Family'] = ((df['Partner'] == 'Yes') | (df['Dependents'] == 'Yes')).astype(int)
    # 高齢者の単身世帯フラグ
    df['Senior_Single'] = ((df['SeniorCitizen'] == 1) & (df['Is_Family'] == 0)).astype(int)
    
    # --- 2. 契約期間 (tenure) ---
    df['Is_New_Customer'] = (df['tenure'] <= 3).astype(int) # 3ヶ月以内の新規顧客
    df['Tenure_Years'] = df['tenure'] / 12.0 # 年単位の契約期間
    
    # --- 3. サービスの契約数 ---
    security_services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
    streaming_services = ['StreamingTV', 'StreamingMovies']
    all_services = security_services + streaming_services
    
    # 'Yes'の数を数える
    df['Count_Security'] = df[security_services].apply(lambda x: (x == 'Yes').sum(), axis=1)
    df['Count_Streaming'] = df[streaming_services].apply(lambda x: (x == 'Yes').sum(), axis=1)
    df['Count_Total_Services'] = df[all_services].apply(lambda x: (x == 'Yes').sum(), axis=1)
    
    # インターネット自体の契約有無
    df['Has_Internet'] = (df['InternetService'] != 'No').astype(int)
    
    # --- 4. 支払いと契約に関するフラグ ---
    # 自動引き落とし（Auto Payment）のフラグ
    df['Is_Auto_Payment'] = df['PaymentMethod'].str.contains('automatic').astype(int)
    # 最も解約率が高い組み合わせ「月額契約 × 電子小切手」
    df['Is_High_Risk_Combo'] = ((df['Contract'] == 'Month-to-month') & (df['PaymentMethod'] == 'Electronic check')).astype(int)
    
    # --- 5. 金額（Charges）の複雑な特徴量 ---
    # 0割りを防ぐため微小な値(1e-5)を足す
    df['Avg_Monthly_Charges'] = df['TotalCharges'] / (df['tenure'] + 1e-5)
    
    # 現在の月額と、過去からの平均月額の差（プラスなら最近プランをアップグレードした可能性）
    df['Charges_Diff_Current_Avg'] = df['MonthlyCharges'] - df['Avg_Monthly_Charges']
    
    # 想定される合計金額と実際の合計金額の差（初期費用、手数料、遅延損害金などによるズレ）
    df['Expected_Total_Charges'] = df['MonthlyCharges'] * df['tenure']
    df['Total_Charges_Diff'] = df['TotalCharges'] - df['Expected_Total_Charges']
    df['Total_Charges_Ratio'] = df['TotalCharges'] / (df['Expected_Total_Charges'] + 1e-5)
    
    # --- 6. カテゴリ変数の交差（Cross Features） ---
    # モデルが捉えやすいように文字列を結合して新しいカテゴリを作る
    df['Contract_Payment_Cross'] = df['Contract'] + "_" + df['PaymentMethod']
    
    return df

# 特徴量エンジニアリングを適用
train_df = advanced_feature_engineering(train_df)
test_df = advanced_feature_engineering(test_df)

# ==========================================
# データの前処理
# ==========================================
train_df["Churn"] = train_df["Churn"].map({'Yes': 1, 'No': 0})

X = train_df.drop(columns=["id", "Churn"])
y = train_df["Churn"]
X_test = test_df.drop(columns=["id", "Churn"], errors='ignore')

# カテゴリ変数の型変換
cat_features = X.select_dtypes(include=['object']).columns.tolist()
for col in cat_features:
    X[col] = X[col].astype('category')
    X_test[col] = X_test[col].astype('category')

# ==========================================
# 5-Fold Cross Validation による学習と予測
# ==========================================
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

test_preds = np.zeros(len(X_test))
oof_preds = np.zeros(len(X))

lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.03,
    'n_estimators': 3000,
    'max_depth': 6,
    'num_leaves': 31,
    'colsample_bytree': 0.8,
    'subsample': 0.8,
    'random_state': 42,
    'n_jobs': -1
}

print("Start Training with Advanced Features...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    model = lgb.LGBMClassifier(**lgb_params)
    
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(500)
        ]
    )
    
    val_pred = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = val_pred
    
    fold_auc = roc_auc_score(y_val, val_pred)
    print(f"Fold {fold+1} AUC: {fold_auc:.5f}")
    
    test_preds += model.predict_proba(X_test)[:, 1] / n_splits

cv_auc = roc_auc_score(y, oof_preds)
print(f"\nOverall CV AUC: {cv_auc:.5f}")

# ==========================================
# 提出用ファイルの作成
# ==========================================
sub_df = pd.DataFrame({
    "id": test_df["id"],
    "Churn": test_preds
})

name = "submission_LGBM_Advanced_FE.csv"
sub_df.to_csv(name, index=False)
print(f"{name}の出力が完了しました")

library import is completed
Data import is completed
Start Training...
[LightGBM] [Info] Number of positive: 107054, number of negative: 368301
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.027363 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1158
[LightGBM] [Info] Number of data points in the train set: 475355, number of used features: 22
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225209 -> initscore=-1.235567
[LightGBM] [Info] Start training from score -1.235567
[200]	valid_0's auc: 0.914165
[400]	valid_0's auc: 0.915295
[600]	valid_0's auc: 0.915644
[800]	valid_0's auc: 0.915843
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[1000]	valid_0's auc: 0.915908
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

,id,Churn
0,594194,0.072497
1,594195,0.000377
2,594196,0.104604
3,594197,0.003410
4,594198,0.511331
5,594199,0.172153
6,594200,0.899212
7,594201,0.002598
8,594202,0.034338
9,594203,0.316814


submission_LGBM_CV_FE.csvの出力が完了しました
